### This ARTICLES notebook 

- Loads journal data into the database  
- Finds the ISSN of Domingo's Incites journals  
- Examines the "completeness" of the Incites journals

- Extracts the works for each journal into the cache

- Load the works into the database after  
    - Filters works into a flat table (work_id, doi, source, host, citation_count etc)  
    - Flattens the authorships table for each work (author_id, institution_id etc)  
    - Filters the reference list to make the cited table (When inverted these are the endogenous citations)  

Note that at this stage the works contain all documents and have not been filtered to articles


In [24]:
import duckdb
import pandas as pd
from pathlib import Path
import diskcache
from itertools import chain

from utils.pandas_setup import pandas_setup
pandas_setup()

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/econommicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')

In [25]:
class SetUp:

    def __init__(self):
        self._setup_db()
        self._setup_cache()
        return
    
    def _setup_db(self):
        self.db = duckdb.connect(MY_DATABASE_FILE)
        self.db.sql("SHOW ALL TABLES").show()
        return
    
    def _setup_cache(self):
        self.cache = diskcache.Cache(MY_CACHE_FILE, size_limit=16_000_000_000)
        print(f'{self.cache.check() = }')
        print(f'{self.cache.volume() = }')
        return
    
    def name_of_global_obj(self, obj=None):
        for objname, oid in globals().items():
            if oid is obj:
                return objname

In [ ]:
class ArticlesETL(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def extract_journals(self):
        # Extract inCites-OpenAlex journal table (ISSN and OA journal_id)
        self.journals = self.db.sql("SELECT * FROM sources_oa_incites").df().\
            rename(columns={'id': 'source_id'}).sort_values('works_count').reset_index()
        print(f'{self.journals.shape = }\n{self.journals.head()}')
        return
    
    def extract_works_by_journal(self):
        # Extract OA works for the journal set, for publication years 2010+ to now
        hold = []
        for row in self.journals.itertuples():
            source_id = row.source_id
            reader = rf'Works().filter(primary_location={{"source": {{"id": "{source_id}"}}}}).filter(publication_year=">2009")'
            if oa := self._read_openalex(reader=reader):
                # print(f'EXTRACTED {len(oa) = } WORKS FOR {row.display_name = }')
                hold.extend(oa)
            else:
                print(f'OpenAlex does not have articles for {source_id = } {row.display_name = }')
            if row.Index % 100 == 0:
                print(f'{row.Index}/{len(self.journals)} completed')
        self._load_articles(hold=hold)                
        return
    
    def _read_openalex(self, reader=None):
        # using the pyalex API call "reader" as key, run the API call, cache the response, and return it (JSON) 
        if result := self.cache.get(reader):
            return result
        try:
            result = list(chain(*eval(reader).paginate(per_page=200)))
        except Exception as e:
            print(f'FAILED TO READ cache or OpenAlex API with {reader = }')
            print(f'{e = }')
            return
        self.cache[reader] = result
        return result
    
    def _load_articles(self, hold=None):
        # convert the downloaded OA works JSON to a dataframe and split out authorships and reference list
        df = pd.DataFrame.from_records(hold)
        if 'is_authors_truncated' not in df.columns:
            df['is_authors_truncated'] = pd.NA
        df = df.rename(columns={'id': 'work_id'}).set_index('work_id')
        print(f'{df.shape = }\n{df.head()}')
        self._load_basic_table(df=df)
        self._load_authorships(df=df)
        self._load_referenced_works(df=df)
        return

    def _load_basic_table(self, df=None):
        # load the database with the flatened and editted works table
        cols = ["doi", "title", "display_name", "publication_year", "primary_location", "type",
                "countries_distinct_count", "institutions_distinct_count", "fwci", "has_fulltext", 
                "cited_by_count", "biblio", "is_retracted", "is_paratext", 
                "referenced_works_count", "cited_by_api_url", "updated_date", "created_date", "is_authors_truncated"]
        df1 = df[cols]
        df_biblio = pd.DataFrame(df1['biblio'].values.tolist(), index=df1.index)
        print(f'{df_biblio.shape = }\n{df_biblio.head()}\n{df_biblio.head()}')

        df2 = pd.DataFrame(df1['primary_location'].values.tolist(), index=df1.index)[['source']]
        df_source = pd.DataFrame(df2['source'].values.tolist(), index=df2.index).\
            rename(columns={'id': 'source_id', 'display_name': 'source_name', 'host_organization': 'host_id', 'host_organization_name': 'host_name'})\
                [['source_id', 'source_name', 'host_id', 'host_name']]
        print(f'{df_source.shape = }\n{df_source.head()}\n{df_source.info()}')
        df3 = pd.concat([df1, df_biblio, df_source], axis=1).drop(columns=['biblio', 'primary_location']).reset_index()
        print(f'{df3.shape = }\n{df3.head()}\n{df3.info()}')
        self.db.sql("CREATE OR REPLACE TABLE works AS (SELECT * FROM df3)")
        self.db.sql("SELECT * FROM works").show()
        self.db.sql("SELECT count(*) FROM works").show()
        return

    def _load_authorships(self, df=None):
        # load database with authorships table after flattening the authors and institutions
        cols = ["authorships"]
        df1 = df[cols]
        df2 = df1.explode('authorships').dropna()
        df3 = pd.DataFrame(df2['authorships'].values.tolist(), index=df2.index)
        df4 = pd.DataFrame(df3['author'].values.tolist(), index=df2.index).rename(columns={'id': 'author_id', 'display_name': 'author_name'})
        df5 = pd.concat([df4, df3], axis=1).set_index(['author_id', 'author_name', 'orcid'], append=True)
        df5 = df5.drop(columns=['author_position', 'author', 'countries', 'is_corresponding', 'raw_author_name', 'raw_affiliation_strings', 'affiliations'])   
        df6 = df5.explode('institutions').dropna()
        df7 = pd.DataFrame(df6['institutions'].values.tolist(), index=df6.index).\
            rename(columns={'id': 'institution_id', 'display_name': 'institution_name'}).drop(columns=['lineage'])
        df7 = df7.reset_index()
        self.db.sql("CREATE OR REPLACE TABLE authorships AS (SELECT * FROM df7)")
        self.db.sql("SELECT * FROM authorships").show()
        self.db.sql("SELECT count(*) FROM authorships").show()
        return

    def _load_referenced_works(self, df=None):
        # load database with referenced works table
        cols = ["referenced_works"]
        ddf = df[cols].reset_index()
        print(ddf.head())
        self.db.sql("CREATE OR REPLACE TABLE cited AS SELECT * FROM ddf")
        self.db.sql("SELECT * FROM cited").show()
        self.db.sql("SELECT count(*) FROM cited").show()
        return
        

In [ ]:
def main():

    jetl = ArticlesETL()
    jetl.extract_journals()
    jetl.extract_works_by_journal()

In [28]:
if __name__ == "__main__":
    main()
    print("DONE!")


┌──────────┬─────────┬────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DONE!
